# UPDATE, DELETE, Transações
---

## Contexto: hora de modificar dados reais

Até aqui o fluxo foi quase todo analítico de selecionar, filtrar, agregar.  
Hoje o cenário muda: o time pediu três tarefas ao mesmo tempo:

1. Reajustar em 10% o preço de todos os produtos de uma categoria
2. Cancelar e remover pedidos antigos com status `"Criado"` há mais de 30 dias
3. Migrar 50.000 registros de um banco legado sem ORM

Essas operações exigem mais cuidado: um `DELETE` mal filtrado apaga o que não devia, e uma atualização no meio de outra pode deixar o banco inconsistente.

Nesta aula você vai:

- Atualizar registros com ORM e com `update()` direto
- Apagar registros respeitando integridade referencial
- Usar transações para garantir operações "tudo ou nada"
- Conhecer o SQLAlchemy Core para operações de alto volume

---

## 1. Configuração

In [1]:
# Bibliotecas necessárias
from pathlib import Path
from datetime import datetime, timedelta
from decimal import Decimal
from sqlalchemy import (
    create_engine, select, and_, or_, func, inspect, text, update,delete,
    String, DateTime, Numeric, Integer, Boolean, ForeignKey, CheckConstraint, Index,
    union, union_all, exists, any_, Table, MetaData
)
from sqlalchemy.orm import (
    DeclarativeBase, Mapped, mapped_column, relationship, Session,
    joinedload, selectinload, aliased
)
from models import Base, Cliente, Produto, ItemPedido, Pedido

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")
# Confirmação visual de que a conexão/engine foi configurada com sucesso
print("Conexão OK ✅")

Conexão OK ✅


## 2. Atualizando registros

Existem duas formas de atualizar com SQLAlchemy. A escolha depende do volume e da lógica envolvida.

### Forma 1 — via ORM (poucos registros, lógica Python)

Carrega o objeto, altera o atributo, faz commit. Simples e legível.

In [2]:
with Session(engine) as session:

    # Cria um produto temporário para os exemplos deste módulo
    produto_tmp = Produto(
        nome="Produto Temporário (M11)",
        categoria="Teste",
        preco_atual=Decimal("100.00"),
        ativo=True,
    )
    session.add(produto_tmp)
    session.commit()
    session.refresh(produto_tmp)
    print(f"Criado  → id: {produto_tmp.id} | preço: R$ {produto_tmp.preco_atual}")

    # Alterar o atributo e commitar — o ORM detecta a mudança automaticamente
    produto_tmp.preco_atual = Decimal("149.90")
    session.commit()

    # session.refresh() sincroniza com o banco — importante quando triggers
    # podem ter alterado valores adicionais após o commit
    session.refresh(produto_tmp)
    print(f"Alterado → preço: R$ {produto_tmp.preco_atual}")

    produto_tmp_id = produto_tmp.id  # salva para usar nas próximas células

Criado  → id: 52 | preço: R$ 100.00
Alterado → preço: R$ 149.90


### Forma 2 — via `update()` direto (muitos registros, sem carregar objetos)

Constrói e executa um `UPDATE` SQL sem instanciar nenhum objeto Python.  
**Muito mais rápido** para centenas ou milhares de registros.

In [3]:
with Session(engine) as session:

    stmt = (
        update(Produto)
        .where(Produto.id == produto_tmp_id)
        .values(categoria="Teste Atualizado")
    )
    result = session.execute(stmt)
    session.commit()

    print(f"Linhas afetadas: {result.rowcount}")

Linhas afetadas: 1


> ⚠️ **UPDATE sem WHERE é desastre.**  
> `update(Produto).values(preco=0)` sem `.where()` zeraria o preço de **todos** os produtos.  
> Sempre:
> 1. Use `.where()` explícito
> 2. Confira `result.rowcount` para ver quantas linhas foram afetadas
> 3. Em caso de dúvida, teste em uma transação e faça rollback antes de commitar

### Atualização em lote usando o bulk_update_mappings()

O `bulk_update_mappings()` é um método do objeto `Session` que facilita o envio de uma lista de dicionários para o banco de dados. Ele é mais "amigável" ao ORM do que o execute(update(...)) puro, embora o comportamento de performance seja similar.


In [4]:
with Session(engine) as session:
    # 1. Criação do lote (Igual ao seu código)
    lote = [
        Produto(nome="Lote A (M11)", categoria="Teste", preco_atual=Decimal("10.00"), ativo=True),
        Produto(nome="Lote B (M11)", categoria="Teste", preco_atual=Decimal("20.00"), ativo=True),
        Produto(nome="Lote C (M11)", categoria="Teste", preco_atual=Decimal("30.00"), ativo=True),
    ]
    session.add_all(lote)
    session.commit()
    
    for p in lote:
        session.refresh(p)

    print("Antes:", [(p.id, float(p.preco_atual)) for p in lote])

    # 2. Preparação dos dados (Dicionários com as chaves PK)
    updates = [
        {"id": lote[0].id, "preco_atual": Decimal("11.11")},
        {"id": lote[1].id, "preco_atual": Decimal("22.22")},
        {"id": lote[2].id, "preco_atual": Decimal("33.33")},
    ]

    session.bulk_update_mappings(Produto, updates)   
    session.commit()

    # O expire_all continua sendo essencial!
    session.expire_all()

    # Verificação (Igual ao seu código)
    novos_precos = session.execute(
        select(Produto.id, Produto.preco_atual).where(Produto.id.in_([p.id for p in lote]))
    ).all()
    
    print("Depois:", [(r.id, float(r.preco_atual)) for r in novos_precos])

    lote_ids = [p.id for p in lote]  # salva para limpeza no próximo passo

Antes: [(53, 10.0), (54, 20.0), (55, 30.0)]
Depois: [(53, 11.11), (54, 22.22), (55, 33.33)]


## 3. Apagando registros

`DELETE` é a operação mais perigosa: remove informação permanentemente.  
Além do filtro errado, há outro risco: **integridade referencial**, por exemplo: você não pode apagar um registro "pai" enquanto existirem filhos apontando para ele.

### Exclusão via ORM

In [5]:
with Session(engine) as session:

    # Apaga o produto individual e os do lote
    produto_para_apagar = session.get(Produto, produto_tmp_id)
    if produto_para_apagar:
        session.delete(produto_para_apagar)

    # Bulk delete dos produtos de lote
    session.execute(delete(Produto).where(Produto.id.in_(lote_ids)))
    session.commit()

    restantes = session.scalar(
        select(func.count()).select_from(Produto).where(Produto.categoria.like("Teste%"))
    )
    print(f"Produtos de teste restantes: {restantes}")

Produtos de teste restantes: 0


### O erro clássico: apagar pai com filhos existentes

Se um `Pedido` tem `ItemPedido` dependentes, tentar apagá-lo viola a integridade referencial, o banco bloqueia.

In [6]:
# Cria pedido + item temporários para demonstrar o erro
from sqlalchemy.exc import IntegrityError

with Session(engine) as session:

    cli  = session.scalars(select(Cliente).limit(1)).first()
    prod = session.scalars(select(Produto).limit(1)).first()

    if not cli or not prod:
        print("Popule o banco com clientes e produtos primeiro.")
    else:
        pedido_tmp = Pedido(
            cliente_id=cli.id,
            data_pedido=datetime.now(),
            status="Criado",
            valor_total=Decimal("0.00"),
        )
        session.add(pedido_tmp)
        session.commit()
        session.refresh(pedido_tmp)

        item_tmp = ItemPedido(
            pedido_id=pedido_tmp.id,
            produto_id=prod.id,
            quantidade=1,
            preco_venda=Decimal("10.00"),
        )
        session.add(item_tmp)
        session.commit()

        pedido_tmp_id = pedido_tmp.id
        print(f"Pedido #{pedido_tmp_id} criado com 1 item.")

        # Tentando apagar o pedido sem apagar os itens primeiro
        try:
            session.delete(pedido_tmp)
            session.commit()
        except IntegrityError:
            session.rollback()
            print("Erro de integridade (esperado): não é possível apagar o pedido com itens vinculados.")
            print("   Solução: apague os filhos antes do pai ou faça tudo na mesma transação.")

Pedido #103 criado com 1 item.
Erro de integridade (esperado): não é possível apagar o pedido com itens vinculados.
   Solução: apague os filhos antes do pai ou faça tudo na mesma transação.


In [7]:
# A forma correta: apagar filhos e pai na mesma transação
with Session(engine) as s:
    with s.begin():  # commit automático se tudo der certo, rollback se der erro
        s.execute(delete(ItemPedido).where(ItemPedido.pedido_id == pedido_tmp_id))
        s.execute(delete(Pedido).where(Pedido.id == pedido_tmp_id))

print("Pedido e itens removidos com sucesso — tudo na mesma transação.")

Pedido e itens removidos com sucesso — tudo na mesma transação.


## 4. Transações: tudo ou nada

Uma **transação** garante que um conjunto de operações seja aplicado de forma **atômica**: ou todas acontecem, ou nenhuma acontece.

```
session.commit()         → confirma tudo desde o último commit
session.rollback()       → desfaz tudo desde o último commit
with session.begin():    → commit automático OU rollback automático em caso de erro
```

In [9]:
# Este evento força o SQLite a respeitar as Foreign Keys

from sqlalchemy import event

@event.listens_for(engine, "connect")
def set_sqlite_pragma(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute("PRAGMA foreign_keys=ON")
    cursor.close()

with Session(engine) as session:
    # Busca um produto real para testar o reajuste
    prod_teste = session.scalars(select(Produto).limit(1)).first()
    preco_antes = prod_teste.preco_atual if prod_teste else None

if prod_teste:
    max_id = 999999  # ID que não existe na tb_clientes

    try:
        # Usamos uma nova sessão para o teste atômico
        with Session(engine) as s:
            with s.begin():  # Início da transação
                # Op 1: Atualiza preço (SQL enviado, mas não commitado)
                s.execute(
                    update(Produto)
                    .where(Produto.id == prod_teste.id)
                    .values(preco_atual=Decimal("9999.99"))
                )

                # Op 2: Insere pedido com cliente inválido
                # Agora, com PRAGMA ON, isso VAI lançar IntegrityError
                s.add(Pedido(
                    cliente_id=max_id,
                    data_pedido=datetime.now(),
                    status="Criado",
                    valor_total=Decimal("0.00"),
                ))
            # Se chegar aqui, faz COMMIT. Se houver erro, faz ROLLBACK automático.

    except IntegrityError:
        print("✅ Sucesso: O erro de integridade foi disparado pelo banco.")

    # Verificação final: O preço deve ter voltado ao original
    with Session(engine) as session:
        # É importante dar refresh ou re-consultar para ver o estado real do banco
        preco_depois = session.scalar(
            select(Produto.preco_atual).where(Produto.id == prod_teste.id)
        )

    print(f"Preço antes:  R$ {preco_antes}")
    print(f"Preço depois: R$ {preco_depois}")

    if preco_antes == preco_depois:
        print("\n✨ O Rollback funcionou! A alteração de preço foi desfeita.")
    else:
        print("\n❌ Atenção: O preço foi alterado mesmo com o erro.")

Preço antes:  R$ 9999.99
Preço depois: R$ 9999.99

✨ O Rollback funcionou! A alteração de preço foi desfeita.


---

## Exercício: Usando IA para isso

**Prompt para gerar um batch update seguro:**
```
Preciso reajustar o preço de produtos em lote usando SQLAlchemy.
Crie um script que:

1. Leia os produtos de uma categoria via select()
2. Aplique um reajuste percentual recebido como parâmetro
3. Valide que nenhum preço novo ultrapassa um teto máximo
4. Execute o batch update com session.execute(update(), lista)
5. Confirme os novos preços com um select() final

Modelo Produto: [cole o modelo]
```
---

### Resposta:Código gerado pelo ChatGPT

In [13]:
def reajustar_precos(engine, categoria, percentual, teto_maximo):
    """
    Reajusta preços de produtos por categoria com validação de teto.
    
    categoria: str
    percentual: ex -> 10 para +10%
    teto_maximo: Decimal
    """

    percentual = Decimal(percentual) / Decimal(100)

    with Session(engine) as session:

        # 1. Ler produtos da categoria
        stmt = select(Produto).where(Produto.categoria == categoria)
        produtos = session.execute(stmt).scalars().all()

        print(f"Produtos encontrados: {len(produtos)}")

        if not produtos:
            print("Nenhum produto encontrado.")
            return

        # 2. Aplicar reajuste e validar teto
        updates = []
        for p in produtos:
            novo_preco = (Decimal(p.preco_atual) * (Decimal(1) + percentual)).quantize(Decimal("0.01"))

            # 3. Validação do teto máximo
            if novo_preco > teto_maximo:
                raise ValueError(
                    f"Preço do produto {p.nome} ultrapassa o teto: {novo_preco} > {teto_maximo}"
                )

            updates.append({
                "id": p.id,
                "preco_atual": novo_preco
            })

        print(f"Atualizações preparadas: {len(updates)}")

        # 4. Batch update
        session.execute(
            update(Produto),
            updates
        )

        session.commit()

        print("Reajuste aplicado com sucesso.")

        # 5. Conferir novos preços
        stmt_final = (
            select(Produto.nome, Produto.preco_atual)
            .where(Produto.categoria == categoria)
            .order_by(Produto.preco_atual.desc())
        )

        novos = session.execute(stmt_final).all()

        print("\nNovos preços:")
        for n in novos[:10]:
            print(f"{n.nome:<25} R$ {n.preco_atual:.2f}")

In [15]:
# Aplicando a função:

reajustar_precos(
    engine,
    categoria="Eletrônicos",
    percentual=10,
    teto_maximo=Decimal("15000.00")
)

Produtos encontrados: 7
Atualizações preparadas: 7
Reajuste aplicado com sucesso.

Novos preços:
Notebook Pro 15           R$ 10999.99
Vitae ipsum               R$ 4588.97
Vero voluptate qui        R$ 3839.62
Reprehenderit voluptate dicta nemo R$ 2855.79
Blanditiis recusandae optio R$ 2115.64
Non doloribus numquam     R$ 2072.26
Inventore occaecati       R$ 1384.74
